In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import load_npz
import random
import random

In [1]:
import pandas as pd

sider = pd.read_csv("./data/sider_lincs_common_clean_FINAL.csv")
drug_index = pd.read_csv("./data/drug_index.csv")

sider_drugs = set(sider["drug_name"])
index_drugs = set(drug_index["drug_id"])

print("Total SIDER drugs:", len(sider_drugs))
print("Total indexed drugs:", len(index_drugs))

missing = sider_drugs - index_drugs
print("Missing drugs:", len(missing))
print("Examples:", list(missing)[:20])

Total SIDER drugs: 699
Total indexed drugs: 679
Missing drugs: 23
Examples: ['Meropenem', 'Westcort', 'norethisterone', 'tetrahydrozoline', 'norepinephrine', 'fluorometholone', 'chloramphenicol', 'chlorthalidone', 'fluoxymesterone', 'zuclopenthixol', 'chlorpromazine', 'podophyllotoxin', 'methylergometrine', 'phenylpropanolamine', 'dexrazoxane', 'desoximetasone', 'Lisinopril', 'clarithromycin', 'Goserelin', 'dihydroergotamine']


In [3]:
adr_index = pd.read_csv("./data/adr_index.csv")

sider_adrs = set(sider["adr_id"])
index_adrs = set(adr_index["adr_id"])

print("Total SIDER ADRs:", len(sider_adrs))
print("Total indexed ADRs:", len(index_adrs))

missing = sider_adrs - index_adrs
print("Missing ADRs:", len(missing))
print("Examples:", list(missing)[:20])

Total SIDER ADRs: 3659
Total indexed ADRs: 3659
Missing ADRs: 0
Examples: []


In [6]:
import numpy as np

Gdrug = np.load("./data/Gdrug_mu_restricted.npy")

print("Matrix rows:", Gdrug.shape[0])
print("Index rows:", len(drug_index))

assert Gdrug.shape[0] == len(drug_index), "Drug index mismatch!"

Matrix rows: 679
Index rows: 679


In [9]:
from scipy.sparse import load_npz

Gadr = load_npz("./data/Gadr_restricted.npz.zip")

print("Matrix rows:", Gadr.shape[0])
print("Index rows:", len(adr_index))

assert Gadr.shape[0] == len(adr_index), "ADR index mismatch!"

Matrix rows: 3659
Index rows: 3659


In [10]:
import random

drug_map = dict(zip(drug_index["drug_id"], drug_index["drug_idx"]))
adr_map = dict(zip(adr_index["adr_id"], adr_index["adr_idx"]))

# pick 5 random samples
samples = sider.sample(5)

for _, row in samples.iterrows():
    d = row["drug_name"]
    a = row["adr_id"]

    if d in drug_map and a in adr_map:
        d_idx = drug_map[d]
        a_idx = adr_map[a]

        print(f"{d} → {d_idx}")
        print(f"{a} → {a_idx}")
        print("-" * 30)

raltitrexed → 535
C0039240 → 354
------------------------------
sulindac → 589
C0013428 → 227
------------------------------
simvastatin → 573
C0028866 → 1789
------------------------------
ivermectin → 321
C0002871 → 178
------------------------------


In [11]:
# duplicates in index
print("Duplicate drugs:", drug_index["drug_id"].duplicated().sum())
print("Duplicate ADRs:", adr_index["adr_id"].duplicated().sum())

Duplicate drugs: 0
Duplicate ADRs: 0


In [12]:
# ensure index is sorted properly
print(drug_index.head())
print(drug_index.tail())

# should match row order of matrix
# i.e., row 0 in matrix = drug_idx 0
assert (drug_index["drug_idx"].values == range(len(drug_index))).all()
assert (adr_index["adr_idx"].values == range(len(adr_index))).all()

   drug_idx        drug_id
0         0    abiraterone
1         1    acamprosate
2         2       acarbose
3         3     acebutolol
4         4  acenocoumarol
     drug_idx       drug_id
674       674   ziprasidone
675       675  zolmitriptan
676       676      zolpidem
677       677    zonisamide
678       678     zopiclone


In [13]:
valid_count = 0

for _, row in sider.iterrows():
    if row["drug_name"] in drug_map and row["adr_id"] in adr_map:
        valid_count += 1

print("Valid pairs:", valid_count)
print("Total pairs:", len(sider))
print("Coverage:", valid_count / len(sider))

Valid pairs: 132945
Total pairs: 137270
Coverage: 0.9684927515116194


In [14]:
assert (drug_index["drug_idx"].values == range(len(drug_index))).all()
assert (adr_index["adr_idx"].values == range(len(adr_index))).all()

In [16]:
Gdrug_mu = pd.read_pickle("./data/Gdrug_mu.pkl")
gene_index= pd.read_csv("./data/gene_index_lincs.csv")

assert list(Gdrug_mu.columns) == list(gene_index["GeneSymbol"])

AssertionError: 